In [1]:
from pathlib import Path
import pandas as pd
from ete3 import NCBITaxa
from Bio import Entrez, SeqIO
from collections import defaultdict
from functools import lru_cache
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import requests

In [3]:
#DATA_DIR = Path("RNAInter_RR_extracted")
#txt_files = list(DATA_DIR.glob("*.txt"))
DATA_PATH = 'RNAInter_RNA_RNA.txt'#txt_files[0]
df = pd.read_csv(DATA_PATH, sep="\t", dtype=str, low_memory=False)

In [4]:
print(f"Number of interactions: {len(df)}") # Display basic info
print("Columns:", df.columns.tolist())      # Show column names
print("NaN in Symbol:", df['Interactor1.Symbol'].isna().sum()) # counting NaNs
print("NaN in Symbol:", df['Interactor2.Symbol'].isna().sum())
print("NaN in Raw_ID1:", df['Raw_ID1'].isna().sum())
print("NaN in Raw_ID1:", df['Raw_ID2'].isna().sum())

Number of interactions: 9484609
Columns: ['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1', 'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2', 'score', 'strong', 'weak', 'predict']
NaN in Symbol: 3
NaN in Symbol: 71
NaN in Raw_ID1: 387389
NaN in Raw_ID1: 97160


In [5]:
exclude_categories = {'Category1', 'Category2', 'unknown', 'unassigned RNA', 'others', 'Protein', 'protein',
                      'PCG',         # Protein-coding Gene
                      'TF',          # transcription factor gene
                      'TR_C_gene',   # T-cell receptor constant region gene; immune-related.
                      'antisense',   # transcripts complementary to sense strands of annotated genes.
                      'TEC',         # “to be experimentally confirmed,” provisional transcripts.
                      'repeats'}     # transcripts overlapping repetitive genomic elements.

# Filter out unwanted categories from both interactors
df_filt = df[~df['Category1'].isin(exclude_categories) & ~df['Category2'].isin(exclude_categories)]
print(f"RNA–RNA interactions (filtered): {len(df_filt)}")
# keep rows where each interactor has at least one identifier
mask_valid_1 = df_filt[['Raw_ID1', 'Interactor1.Symbol']].notna().any(axis=1)
mask_valid_2 = df_filt[['Raw_ID2', 'Interactor2.Symbol']].notna().any(axis=1)
df_filt = df_filt[mask_valid_1 & mask_valid_2]
print(f"RNA–RNA interactions (filtered | identifiable): {len(df_filt)}")

RNA–RNA interactions (filtered): 9429474
RNA–RNA interactions (filtered | identifiable): 9429405


In [6]:
parsed = []
def parse_ids(id_value):
    if pd.isna(id_value):
        return [None]
    elif ':' in id_value:
        prefix, ids_str = id_value.split(':', 1)
        return [f"{prefix}:{id_.strip()}" for id_ in ids_str.split(';')]
    else:
        parsed.append(id_value)
        return [id_value.strip()]

# Add list columns
df_filt['Raw_ID1_list'] = df_filt['Raw_ID1'].apply(parse_ids)
df_filt['Raw_ID2_list'] = df_filt['Raw_ID2'].apply(parse_ids)

In [7]:
# Standardise the species to TAXIDs
# 1. See unique species
df_filt.replace({'-': None}, inplace=True)
unique_species = pd.unique(df_filt[['Species1', 'Species2']].values.ravel())
print("Unique species:", len(unique_species))

ncbi = NCBITaxa()
unique_species_taxids = {}
missing = []

for test_species in unique_species:
    # Skip placeholders
    if test_species in [None]: continue

    # Handle known special cases first
    if test_species == 'Solanum tuberosum L.':
        unique_species_taxids[test_species] = "4113"
        continue
    if test_species == 'Rhodobacter sphaeroides ATCC 17025':
        unique_species_taxids[test_species] = "1061"
        continue
    if test_species == 'Luffa cylindrica':
        unique_species_taxids[test_species] = "3669"
        continue
    # Query NCBI
    try:
        taxid = str(ncbi.get_name_translator([test_species])[test_species][0])
        unique_species_taxids[test_species] = taxid
    except KeyError:
        missing.append(test_species)

print("Mapped species (unique -1 for None):", len(unique_species_taxids))
print("Missing species:", missing)

Unique species: 135
Mapped species (unique -1 for None): 134
Missing species: []


In [8]:
df_filt['Species1_TaxID'] = df_filt['Species1'].map(unique_species_taxids)
df_filt['Species2_TaxID'] = df_filt['Species2'].map(unique_species_taxids)

unmapped_s1 = df_filt.loc[df_filt['Species1'].notna() & df_filt['Species1_TaxID'].isna(), 'Species1'].unique()
unmapped_s2 = df_filt.loc[df_filt['Species2'].notna() & df_filt['Species2_TaxID'].isna(), 'Species2'].unique()

print("Unmapped Species1:", unmapped_s1)
print("Unmapped Species2:", unmapped_s2)

# All is correctly mapping and deleting NaN species
df_filt = df_filt.dropna(subset=['Species1_TaxID', 'Species2_TaxID'])
print(f"RNA–RNA interactions: {len(df_filt)}")

Unmapped Species1: []
Unmapped Species2: []
RNA–RNA interactions: 9429404


# mRNA cases

In [9]:
df_mrna = df_filt[(df_filt["Category1"] == "mRNA") | (df_filt["Category2"] == "mRNA")].copy()
print("mRNA-involved interactions:", len(df_mrna))
df_mrna.head(5)

mRNA-involved interactions: 9224109


,RNAInterID,Interactor1.Symbol,Category1,Species1,Interactor2.Symbol,Category2,Species2,Raw_ID1,Raw_ID2,score,strong,weak,predict,Raw_ID1_list,Raw_ID2_list,Species1_TaxID,Species2_TaxID
10225,RR01511940,U1,snRNA,Homo sapiens,ZNF536,mRNA,Homo sapiens,Ensembl:ENSG00000277918,NCBI:9745,0.2566,NaN,PARIS,NaN,[Ensembl:ENSG00000277918],[NCBI:9745],9606,9606
10226,RR01511934,U1,snRNA,Homo sapiens,ZNF536,mRNA,Homo sapiens,Ensembl:ENSG00000274210,NCBI:9745,0.2566,NaN,PARIS,NaN,[Ensembl:ENSG00000274210],[NCBI:9745],9606,9606
10227,RR01511929,U1,snRNA,Homo sapiens,ZNF536,mRNA,Homo sapiens,Ensembl:ENSG00000270722,NCBI:9745,0.2566,NaN,PARIS,NaN,[Ensembl:ENSG00000270722],[NCBI:9745],9606,9606
10228,RR01512781,U2,snRNA,Homo sapiens,ZNF536,mRNA,Homo sapiens,Ensembl:ENSG00000275219,NCBI:9745,0.2566,NaN,PARIS,NaN,[Ensembl:ENSG00000275219],[NCBI:9745],9606,9606
10229,RR01512774,U2,snRNA,Homo sapiens,ZNF536,mRNA,Homo sapiens,Ensembl:ENSG00000273694,NCBI:9745,0.2566,NaN,PARIS,NaN,[Ensembl:ENSG00000273694],[NCBI:9745],9606,9606


In [10]:
def normalize_id_list_keep_na(x):
    if not isinstance(x, list): return pd.NA
    cleaned = [str(i) for i in x if i is not None and str(i).strip() not in {"", "-", "nan", "None"}]
    if len(cleaned) == 0: return pd.NA
    
    return tuple(sorted(cleaned))

df_mrna['Raw_ID1_key'] = df_mrna['Raw_ID1_list'].apply(normalize_id_list_keep_na)
df_mrna['Raw_ID2_key'] = df_mrna['Raw_ID2_list'].apply(normalize_id_list_keep_na)

# Only take entities where Category == mRNA
mrna_entities_1 = df_mrna[df_mrna["Category1"] == "mRNA"][['Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_key']]
mrna_entities_2 = df_mrna[df_mrna["Category2"] == "mRNA"][['Interactor2.Symbol', 'Category2', 'Species2_TaxID', 'Raw_ID2_key']]

mrna_entities = (
    pd.concat([
        mrna_entities_1.rename(columns={'Interactor1.Symbol': 'Symbol', 'Category1': 'Category', 'Species1_TaxID': 'Species', 'Raw_ID1_key': 'Raw_ID_key'}),
        mrna_entities_2.rename(columns={'Interactor2.Symbol': 'Symbol', 'Category2': 'Category', 'Species2_TaxID': 'Species', 'Raw_ID2_key': 'Raw_ID_key'
        })], ignore_index=True).drop_duplicates())

print("Unique mRNA entities:", len(mrna_entities))
print("NaN in Raw_ID_key:", mrna_entities['Raw_ID_key'].isna().sum())
print("NaN in Symbol:", mrna_entities['Symbol'].isna().sum())
print("Species involved:", mrna_entities['Species'].nunique())

Unique mRNA entities: 213723
NaN in Raw_ID_key: 10329
NaN in Symbol: 1
Species involved: 111


In [11]:
# Statistic for the sources of mRNA entities
sources = (mrna_entities["Raw_ID_key"].astype(str).str.replace(r"[(),'\"]", "", regex=True).str.split(":").str[0])  # Source
source_counts = sources.value_counts(dropna=False)
print(source_counts)

Raw_ID_key
NCBI       202217
<NA>        10329
Ensembl      1177
Name: count, dtype: int64


# Searching for sequences

### 1- 202.217 NCBI IDs (~94%)

In [12]:
# Obtain the GeneID just only for the NCBI source
df_ncbi = mrna_entities[mrna_entities["Raw_ID_key"].astype(str).str.contains("NCBI:", na=False)].copy()
# Extract numeric GeneID
df_ncbi["GeneID"] = (df_ncbi["Raw_ID_key"].astype(str).str.extract(r"NCBI:(\d+)", expand=False).astype("Int64"))
print("Number of mRNA in NCBI:", len(df_ncbi))    
df_ncbi.head()

Number of mRNA in NCBI: 202217


,Symbol,Category,Species,Raw_ID_key,GeneID
0,0610009B22Rik,mRNA,10090,"(NCBI:66050,)",66050
1,0610010F05Rik,mRNA,10090,"(NCBI:71675,)",71675
9,0610010K14Rik,mRNA,10090,"(NCBI:104457,)",104457
20,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676
21,0610012G03Rik,mRNA,10090,"(NCBI:106264,)",106264


In [13]:
print("Number of mRNA in NCBI:", len(df_ncbi)) 
print("Unique Symbols:", df_ncbi["Symbol"].nunique(), "but duplicates by symbol:", len(df_ncbi)-df_ncbi["Symbol"].nunique())
print("Unique GeneIDs:", df_ncbi["GeneID"].nunique(), "but duplicates by GeneID:", len(df_ncbi)-df_ncbi["GeneID"].nunique())

# Some symbols are just older names of the same gene, but some GeneIDs are duplicated with different symbols.
# But the 202.217 are unique in the combination of Symbol + GeneID.
# search using GeneID as primary source, future analisys: merging that with the same Symbol.

Number of mRNA in NCBI: 202217
Unique Symbols: 144444 but duplicates by symbol: 57773
Unique GeneIDs: 198496 but duplicates by GeneID: 3721


In [14]:
# --- 1) Load gene2refseq fast (vectorized) ---
g2r = pd.read_csv(
    "gene2refseq",
    sep="\t", comment="#", header=None,
    usecols=[0, 1, 2, 3, 15],   # tax_id, GeneID, status, RNA_accession, Symbol
    names=["taxid", "GeneID", "status", "rna_acc", "Symbol"], dtype=str, low_memory=False
)
# keep mRNA RefSeq
g2r = g2r[(g2r["status"] != "SUPPRESSED") & (g2r["rna_acc"] != "-") & (g2r["rna_acc"].str.startswith(("NM_", "XM_"), na=False))].copy()

In [15]:
# --- 2) Collapse to (taxid, GeneID) -> list of RefSeq mRNA transcripts --- SEARCHING JUST BY NCBI IDs AND SPECIES
refseq_map = (
    g2r.groupby(["taxid", "GeneID"])["rna_acc"]
      .agg(lambda s: sorted(set(s))).reset_index().rename(columns={"rna_acc": "refseq"}))

# --- 3) Merge onto your df_ncbi (Species == taxid) ---
df_ncbi["Species"] = df_ncbi["Species"].astype(str)
df_ncbi["GeneID"]  = df_ncbi["GeneID"].astype(str)

df_ncbi = df_ncbi.merge(refseq_map, left_on=["Species", "GeneID"], right_on=["taxid", "GeneID"], how="left").drop(columns=["taxid"])
df_ncbi.head()

,Symbol,Category,Species,Raw_ID_key,GeneID,refseq
0,0610009B22Rik,mRNA,10090,"(NCBI:66050,)",66050,"[NM_001417819.1, NM_001417820.1, NM_001417822...."
1,0610010F05Rik,mRNA,10090,"(NCBI:71675,)",71675,"[NM_001347242.1, NM_027860.3, XM_006514808.4, ..."
2,0610010K14Rik,mRNA,10090,"(NCBI:104457,)",104457,"[NM_001177601.1, NM_001177603.1, NM_001177606...."
3,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676,NaN
4,0610012G03Rik,mRNA,10090,"(NCBI:106264,)",106264,[NM_134107.1]


In [14]:
print("Number of refseq:", df_ncbi["refseq"].notna().sum(),"/",len(df_ncbi), "Percentage:", 100*df_ncbi["refseq"].notna().sum()/len(df_ncbi))

Number of refseq: 194559 / 202217 Percentage: 96.21297912638403


###  BREAK TO SAVE DATAFRAME AND GET SEQUENCES (AND GO TERMS 

In [17]:
def count_sequences(x):
    if isinstance(x, list): return len(x)
    else: return 0

In [19]:
# Create df_ncbi_194k deleting the refseq column with NaN values (where I didnt found any refseq) and then save as a csv
df_ncbi_194k = df_ncbi.dropna(subset=["refseq"]).copy() 
df_ncbi_194k["refseq_count"] = df_ncbi_194k["refseq"].apply(count_sequences)
df_ncbi_194k.to_csv("df_ncbi_194k.csv", index=False)                             # Just 26 kB

In [20]:
# Load the df_ncbi_194k.csv to use directly without having to run the previous code again
df_ncbi_194k = pd.read_csv("df_ncbi_194k.csv")
print("Number of mRNA with RefSeq:", len(df_ncbi_194k))
print("Total number of RefSeq transcripts:", df_ncbi_194k["refseq_count"].sum()) # Count how many RefSeq transcripts we have in total (sum of refseq_count)
df_ncbi_194k.head(3)

Number of mRNA with RefSeq: 194559
Total number of RefSeq transcripts: 964784


,Symbol,Category,Species,Raw_ID_key,GeneID,refseq,refseq_count
0,0610009B22Rik,mRNA,10090,"('NCBI:66050',)",66050,"['NM_001417819.1', 'NM_001417820.1', 'NM_00141...",5
1,0610010F05Rik,mRNA,10090,"('NCBI:71675',)",71675,"['NM_001347242.1', 'NM_027860.3', 'XM_00651480...",12
2,0610010K14Rik,mRNA,10090,"('NCBI:104457',)",104457,"['NM_001177601.1', 'NM_001177603.1', 'NM_00117...",8


In [48]:
print("Number of mRNA in NCBI:", len(df_ncbi_194k)) 
print("Unique Symbols:", df_ncbi_194k["Symbol"].nunique(), "but duplicates by symbol:", len(df_ncbi_194k)-df_ncbi_194k["Symbol"].nunique())
print("Unique GeneIDs:", df_ncbi_194k["GeneID"].nunique(), "but duplicates by GeneID:", len(df_ncbi_194k)-df_ncbi_194k["GeneID"].nunique())

Number of mRNA in NCBI: 194559
Unique Symbols: 137655 but duplicates by symbol: 56904
Unique GeneIDs: 191013 but duplicates by GeneID: 3546


In [ ]:
# First we need to decide what to do with the duplicated GeneIDs and Symbols, or just filter by symbol after getting the RefSeqs. 

In [15]:
# Marco's Question: If for an mRNA I get all its RefSeq related to, is there a way to get its GO terms?
#           1. GeneID (Entrez Gene) → GO terms? First attempt
#           2. RefSeq transcript accession → GO terms?

In [44]:
# UPDATED code for parallelization

# MyGene.Info provides simple-to-use REST web services to query/retrieve gene annotation data. Find GO terms by NCBI GeneID.
# Im going to check for more options later, but for now I will try the /v3/gene endpoint with POST (robust for batches).
# Code trusting in GeneID.

# Standard annotation practice: GO terms are attached to the gene / gene product, not to each RefSeq transcript independently.


import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# --- Cache for GO terms ---
go_cache = {}  # GeneID -> {"BP": [...], "MF": [...], "CC": [...]}

def normalize_go(block):
    if not block:
        return []
    if isinstance(block, dict):
        block = [block]
    return [
        {"id": x.get("id"), "term": x.get("term"), "evidence": x.get("evidence")}
        for x in block
        if x.get("id") and x.get("term")
    ]

def fetch_go_batch(ids):
    """ Fetch GO annotations for a batch of GeneIDs using the batched POST endpoint. """
    if not ids:
        return
    # send batch request
    try:
        r = requests.post(
            "https://mygene.info/v3/gene",
            data={"ids": ",".join(map(str, ids)), "fields": "go"},
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=60
        )
        r.raise_for_status()
    except Exception as e:
        print(f"Batch request failed for {ids[:5]}..: {e}")
        return

    # populate cache
    resp = r.json()
    for rec in resp:
        # ensure we get the ID in a consistent key
        gid = str(rec.get("entrezgene") or rec.get("_id"))
        go = rec.get("go") or {}
        go_cache[gid] = {
            "BP": normalize_go(go.get("BP")),
            "MF": normalize_go(go.get("MF")),
            "CC": normalize_go(go.get("CC"))
        }

# --- Apply to DataFrame subset ---
print("Start:", datetime.now())
test_subset = df_ncbi_194k #.head(10000).copy()
unique_genes = [str(int(x)) for x in test_subset["GeneID"].dropna().unique()]

# --- Set up batch parameters ---
batch_size = 1000
batches = [unique_genes[i : i + batch_size] for i in range(0, len(unique_genes), batch_size)]

# Parallel batch processing
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(fetch_go_batch, batch) for batch in batches]
    for future in as_completed(futures):
        # no return value, just ensure completion
        future.result()

print("Fetched batches:", datetime.now())

# --- Map results back to dataframe ---
test_subset["GO"] = test_subset["GeneID"].map(
    lambda g: go_cache.get(str(int(g)), {"BP": [], "MF": [], "CC": []})
)
test_subset["GO_BP_count"] = test_subset["GO"].apply(lambda d: len(d["BP"]))
test_subset["GO_MF_count"] = test_subset["GO"].apply(lambda d: len(d["MF"]))
test_subset["GO_CC_count"] = test_subset["GO"].apply(lambda d: len(d["CC"]))

print("Done:", datetime.now())

Start: 2026-03-06 16:46:29.107820
Fetched batches: 2026-03-06 16:49:09.392913
Done: 2026-03-06 16:49:09.989914


In [45]:
print("GO BP terms counts:", test_subset["GO_BP_count"].sum())
print("GO MF terms counts:", test_subset["GO_MF_count"].sum())
print("GO CC terms counts:", test_subset["GO_CC_count"].sum())
# Numbers of refseq and GO terms are going to be crazy but important if we check correclty to be honest.

GO BP terms counts: 1003704
GO MF terms counts: 673342
GO CC terms counts: 730160


In [47]:
test_subset['GO'].iloc[0]

{'BP': [{'id': 'GO:0001501',
   'term': 'skeletal system development',
   'evidence': 'ISO'},
  {'id': 'GO:0006888',
   'term': 'endoplasmic reticulum to Golgi vesicle-mediated transport',
   'evidence': 'IBA'},
  {'id': 'GO:0006888',
   'term': 'endoplasmic reticulum to Golgi vesicle-mediated transport',
   'evidence': 'IEA'},
  {'id': 'GO:0016192',
   'term': 'vesicle-mediated transport',
   'evidence': 'IEA'}],
 'MF': [{'id': 'GO:0044325',
   'term': 'transmembrane transporter binding',
   'evidence': 'ISO'}],
 'CC': [{'id': 'GO:0005634', 'term': 'nucleus', 'evidence': 'IBA'},
  {'id': 'GO:0005634', 'term': 'nucleus', 'evidence': 'ISO'},
  {'id': 'GO:0005737', 'term': 'cytoplasm', 'evidence': 'IBA'},
  {'id': 'GO:0005783', 'term': 'endoplasmic reticulum', 'evidence': 'ISO'},
  {'id': 'GO:0005793',
   'term': 'endoplasmic reticulum-Golgi intermediate compartment',
   'evidence': 'IEA'},
  {'id': 'GO:0030008', 'term': 'TRAPP complex', 'evidence': 'IBA'},
  {'id': 'GO:0030008', 'term':

#### CONTINUATION ##########

In [ ]:
'''################ I dont believe more in this part, is searching but using the Symbol instead of the GeneID. ################

# Using Symbols instead of GeneIDs to recover more RefSeq accessions for the NaN subset (7.658 rows)
# 1) Take the NaN subset: Contains 7.658 rows
df_ncbi2 = df_ncbi.copy()
df_nan = df_ncbi2[df_ncbi2["refseq"].isna()].copy()


need = (df_nan[["Species", "Symbol"]].dropna().drop_duplicates().rename(columns={"Species": "taxid"}).astype(str)) # 2) Create need dataframe
g2r_need = g2r.merge(need, on=["taxid", "Symbol"], how="inner") # 3) Restrict g2r to only those pairs

# 4) Build symbol -> refseq list for needed SYMBOLS 
refseq_map = (
    g2r_need.groupby(["taxid", "Symbol"])["rna_acc"]
    .agg(lambda s: sorted(set(s)))
    .rename("refseq")
)

# 5) Map back using index matching
for idx, row in df_nan.iterrows():
    key = (str(row["Species"]), row["Symbol"])
    if key in refseq_map.index:
        df_ncbi2.loc[idx, "refseq"] = refseq_map[key]

# 6) Print result
print("Number of refseq:", df_ncbi2["refseq"].notna().sum(),"/",len(df_ncbi2), "Percentage:", 100*df_ncbi2["refseq"].notna().sum()/len(df_ncbi2))

'''

Number of refseq: 195540 / 202217 Percentage: 96.69810154438053


In [21]:
### Some statistics about the mRNA entities
# Ensure refseq column is treated as list where possible
def count_sequences(x):
    if isinstance(x, list): return len(x)
    else: return 0
    
df_ncbi2["refseq_count"] = df_ncbi2["refseq"].apply(count_sequences)

# Bins (the most easiest way)
bins = {"1": 0,"2": 0,"3": 0,"3-5": 0,"5-10": 0,">10": 0}
for count in df_ncbi2["refseq_count"]:
    if count == 1: bins["1"] += 1
    elif count == 2: bins["2"] += 1
    elif count == 3: bins["3"] += 1
    elif 4 <= count <= 5: bins["3-5"] += 1
    elif 6 <= count <= 10: bins["5-10"] += 1
    elif count > 10: bins[">10"] += 1

labels, values = list(bins.keys()), list(bins.values()) # Convert to plotting format

# Print statistics
print("RefSeq count distribution:")
for k, v in bins.items(): print(f"{k} sequences: {v} genes")

# Plot vertical bar chart
plt.figure()
plt.bar(labels, values)
plt.xlabel("Number of RefSeq sequences per Gene")
plt.ylabel("Number of Genes")
plt.title("Distribution of RefSeq Sequences per Gene")
plt.show()

NameError: name 'df_ncbi2' is not defined

In [118]:
# and just doing copy to dont lose the previous dataframe for now
df_ncbi3 = df_ncbi2.copy()
df_nan2 = df_ncbi3[df_ncbi3["refseq"].isna()].copy()
df_nan2

,Symbol,Category,Species,Raw_ID_key,GeneID,refseq,refseq_count
3,0610012D04Rik,mRNA,10090,"(NCBI:71676,)",71676,NaN,0
9,1110003F10Rik,mRNA,10090,"(NCBI:68476,)",68476,NaN,0
20,1110059G02Rik,mRNA,10090,"(NCBI:68786,)",68786,NaN,0
27,1500002K03Rik,mRNA,10090,"(NCBI:76682,)",76682,NaN,0
28,1500005C15Rik,mRNA,10090,"(NCBI:72019,)",72019,NaN,0
...,...,...,...,...,...,...,...
201898,znf1158,mRNA,7955,"(NCBI:108181161,)",108181161,NaN,0
201903,znf1168,mRNA,7955,"(NCBI:101883958,)",101883958,NaN,0
201904,znf1172,mRNA,7955,"(NCBI:101884237,)",101884237,NaN,0
201908,znf1178,mRNA,7955,"(NCBI:103909503,)",103909503,NaN,0


I tried to use the API but results was ???

so Im just working with the gene2refseq (idk how you do changes between differents IDs or symbols)

### 2- 10.329 <NA> Symbol IDs (~4%)

In [150]:
# Obtain the cases without source identifiers (NaN in Raw_ID_key)
df_NA = mrna_entities[mrna_entities["Raw_ID_key"].isna()].copy()
df_NA.head(5)
# Here, just what the hell are that symbols

,Symbol,Category,Species,Raw_ID_key
192,14-3-3ÃÂ,mRNA,9606,<NA>
1092,18S_RRNA,mRNA,9606,<NA>
1865,28S_RRNA,mRNA,9606,<NA>
3125,7SLRNA,mRNA,9606,<NA>
47275,ArpH,mRNA,99287,<NA>


In [152]:
# 2) Build needed (taxid, Symbol) pairs
need = (df_NA[["Species", "Symbol"]].dropna().drop_duplicates().rename(columns={"Species": "taxid"}).astype(str))

# 3) Restrict gene2refseq to only needed pairs
g2r_need = g2r.merge(need, on=["taxid", "Symbol"], how="inner")

# 4) Build mapping (taxid, Symbol) -> list of mRNA RefSeq
refseq_df = (g2r_need.groupby(["taxid", "Symbol"])["rna_acc"].agg(lambda s: sorted(set(s))).reset_index(name="refseq"))

# 5) Merge back to df_NA
df_NA["Species"] = df_NA["Species"].astype(str)

df_NA = df_NA.merge(
    refseq_df,
    left_on=["Species", "Symbol"],
    right_on=["taxid", "Symbol"],
    how="left"
).drop(columns=["taxid"])

# 6) Report recovery
print("Number of refseq:", df_NA["refseq"].notna().sum(), "/", len(df_NA))

Number of refseq: 1649 / 10329


### 3-1.177 Ensemble IDs (~0.55%)

In [136]:
df_ENS = mrna_entities[mrna_entities["Raw_ID_key"].astype(str).str.contains("Ensembl:", na=False)].copy()
df_ENS["Ensembl_ID"] = (df_ENS["Raw_ID_key"].astype(str).str.extract(r"Ensembl:([^,\)]+)", expand=False))
print("Number of mRNA in Ensembl:", len(df_ENS))
df_ENS["Ensembl_ID"] = df_ENS["Ensembl_ID"].str.rstrip("'")
df_ENS.head()

## gene2refseq have some Ensembl IDs to try to retrieve refseq

Number of mRNA in Ensembl: 1177


,Symbol,Category,Species,Raw_ID_key,Ensembl_ID
934,1700094D03Rik,mRNA,10090,"(Ensembl:ENSMUSG00000078667,)",ENSMUSG00000078667
1111,2010015L04Rik,mRNA,10090,"(Ensembl:ENSMUSG00000042233,)",ENSMUSG00000042233
1534,2410017P09Rik,mRNA,10090,"(Ensembl:ENSMUSG00000073240,)",ENSMUSG00000073240
1752,2610524H06Rik,mRNA,10090,"(Ensembl:ENSMUSG00000092486,)",ENSMUSG00000092486
1856,2810453I06Rik,mRNA,10090,"(Ensembl:ENSMUSG00000048910,)",ENSMUSG00000048910


In [153]:
# 2) Build needed (taxid, Symbol) pairs
need = (df_ENS[["Species", "Symbol"]].dropna().drop_duplicates().rename(columns={"Species": "taxid"}).astype(str))

# 3) Restrict gene2refseq to only needed pairs
g2r_need = g2r.merge(need, on=["taxid", "Symbol"], how="inner")

# 4) Build mapping (taxid, Symbol) -> list of mRNA RefSeq
refseq_df = (g2r_need.groupby(["taxid", "Symbol"])["rna_acc"].agg(lambda s: sorted(set(s))).reset_index(name="refseq"))

# 5) Merge back to df_NA
df_ENS["Species"] = df_ENS["Species"].astype(str)

df_ENS = df_ENS.merge(
    refseq_df,
    left_on=["Species", "Symbol"],
    right_on=["taxid", "Symbol"],
    how="left"
).drop(columns=["taxid"])

# 6) Report recovery
print("Number of refseq:", df_ENS["refseq"].notna().sum(), "/", len(df_ENS))

Number of refseq: 213 / 1177


# RESULTS

### Summary of results:
- mRNA-involved interactions: 9.224.109
- Unique mRNA entities: 213.723
### Source of the IDs | RefSeq Retrieved (for now and just using gene2refseq and nothing else)
- NCBI       202.217  |       195.540    #(searching with IDs and symbols) 
- NA          10.329  |        1.649     #(searching just with symbol)
- Ensembl      1.177  |         213      #(searching just with symbol)  
- Total      213.723  |       197.402    #(92.36% of all the sequences has retrieved at least one refseq)

### For NCBI IDs: Number of refseq: 195.540 / 202.217 Percentage: ~96.7%

### RefSeq count distribution:
- 1 sequences: 88.424 genes
- 2 sequences: 30.802 genes
- 3 sequences: 15.265 genes
- 3-5 sequences: 21.150 genes
- 5-10 sequences: 18.724 genes
- >10 sequences: 21.175 genes